# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"\nVersion: {metadata.version}\nAuthors: {getattr(metadata, 'author', 'N/A')}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

Let's print information about all record sets in this dataset. **For each recordset, we'll list its `@id` and all available fields (by their `@id` and name)**.

In [ ]:
# List all record sets and their fields using @id references
record_sets = dataset.metadata.recordSet  # List of recordset objects
if not record_sets:
    print("No record sets found in metadata. Loading available files as record sets.")
    # Some Croissant schemas list distributions as data
    if hasattr(metadata, "distribution"):
        print("Distributions found:")
        for dist in metadata.distribution:
            dist_id = getattr(dist, '@id', dist)
            print(f"  Distribution @id: {dist_id}")
else:
    for idx, record_set in enumerate(record_sets):
        rid = getattr(record_set, '@id', f"[Unknown-{idx}]")
        print(f"RecordSet @id: {rid}, name: {getattr(record_set, 'name', '[no name]')}")
        fields = getattr(record_set, 'field', [])
        for field in fields:
            fid = getattr(field, '@id', field)
            fname = getattr(field, 'name', '[no name]')
            print(f"    Field @id: {fid:40} | name: {fname}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We'll use the main tabular file for this dataset (proteins table) referenced by its `@id`. For this schema, let's examine the available distributions (data files) and their columns.

In [ ]:
# If there are no explicit record sets, attempt to use the main tabular DataFileObject in distribution
dist_ids = []
if hasattr(metadata, "distribution"):
    # If multiple distributions, inspect them
    for dist in metadata.distribution:
        dist_ids.append(getattr(dist, '@id', dist))
print(f"Available Distributions (possible record sets):\n{dist_ids}")

# Pick the first distribution as default for this example
main_recordset_id = dist_ids[0]

# Load data for each available record set (using @id)
dataframes = {}
for rid in dist_ids:
    print(f"Loading records from record set/distribution @id: {rid}")
    try:
        records = list(dataset.records(record_set=rid))
        df = pd.DataFrame(records)
        dataframes[rid] = df
        print(f"Loaded DataFrame with {df.shape[0]} rows and {df.shape[1]} columns.")
    except Exception as e:
        print(f"  Failed to load records for {rid}: {e}")

# Show the columns of the main record set
print(f"\nColumns in main record set (@id: {main_recordset_id}):\n{list(dataframes[main_recordset_id].columns)}")
dataframes[main_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

Let's inspect numeric fields (e.g., coverage percentage or molecular weight) and perform transformations as examples. All columns are referenced with their names (since they correspond to their `@id` or are unique in the file metadata).

In [ ]:
# Pick representative numeric and categorical columns. Use @id/names as shown in previous output.
# (You may need to adjust the field name to one in the actual columns. Common choices: "MW_kDa", "Coverage_percent", "Peptide_Count", etc)
cols = list(dataframes[main_recordset_id].columns)
print(f"Available columns: {cols}")

# Use 'MW_kDa' if present, else pick other numeric
numeric_field = None
for candidate in ['MW_kDa', 'MolecularWeight', 'Coverage_percent', 'Peptide_Count']:
    if candidate in cols:
        numeric_field = candidate
        break
if numeric_field is None:
    # Just pick the first numeric-looking column
    for c in cols:
        if dataframes[main_recordset_id][c].dtype in ['int64', 'float64']:
            numeric_field = c
            break

# Use a group field, e.g., 'Sample' or 'Group' or a string field
group_field = None
for candidate in ['Sample', 'Condition', 'Group', 'Experiment']:
    if candidate in cols:
        group_field = candidate
        break

if numeric_field is None:
    print("No numeric field available for EDA.")
else:
    print(f"Using numeric field: {numeric_field}")

    # Convert to numeric, coercing errors
    df = dataframes[main_recordset_id].copy()
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    threshold = df[numeric_field].mean()  # Use mean as example
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping
    if group_field is not None and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field].mean()
        print(f"\nGrouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion
This notebook demonstrated how to:
- Load and explore Croissant metadata and records with `mlcroissant`
- Reference all entities (`recordSet`, `field`, etc) by their `@id`
- Load records into DataFrames and overview their columns
- Apply basic EDA (filtering, normalization, grouping) and visualize distributions

For further exploration, consult the full [FAIR² dataset documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa) or inspect other record sets and fields by their `@id`

**Remember**: Fields and columns may differ based on schema updates; always reference entities using their `@id` or column names as defined in the Croissant file.